# Synthetic Signal Visualization

Use this notebook to quickly inspect the built-in synthetic signals. The core plotting logic lives in `signal_processing_prep.plotting`; the notebook only calls package functions.

## Setup

For interactive sliders in VS Code or Jupyter, install the notebook extra from the repository root:

```powershell
python -m pip install -e ".[dev,notebook]"
```

The first code cell tries to enable the interactive matplotlib widget backend and falls back to inline plots if that backend is unavailable.

In [ ]:
from IPython import get_ipython

ip = get_ipython()
if ip is None:
    print("No IPython kernel detected; cannot configure matplotlib backend.")
else:
    try:
        ip.run_line_magic("matplotlib", "widget")
        print("Configured matplotlib for interactive widget backend.")
    except Exception:
        try:
            ip.run_line_magic("matplotlib", "inline")
            print("Interactive widget backend unavailable; using inline backend.")
        except Exception:
            print("Failed to configure matplotlib backend; proceeding with default settings.")

In [ ]:
import matplotlib.pyplot as plt
from signal_processing_prep.plotting import plot_time_signal, plot_time_signal_navigator, plot_time_signal_adaptive
from signal_processing_prep.synthetic import (
    add_signals,
    chirp_signal,
    clipped_signal,
    convolve_signals,
    impulse_train,
    make_synthetic_dataset,
    multiply_signals,
    noisy_sine_wave,
    sine_wave,
    transient_burst,
    window_signal,
)

## Available Synthetic Records

In [ ]:
records = make_synthetic_dataset()
for record in records:
    print(
        f"{record.name:22s} label={record.label:16s} "
        f"samples={record.n_samples:5d} duration={record.duration_seconds:.3f}s "
        f"fs={record.sampling_rate_hz:g}Hz"
    )


## Custom Signal Composition

Build custom synthetic examples by combining standard signals with addition, multiplication, and convolution.

In [ ]:
sampling_rate_hz = 4000.0
duration_seconds = 2.0

low_tone = sine_wave(
    frequency_hz=35.0,
    duration_seconds=duration_seconds,
    sampling_rate_hz=sampling_rate_hz,
    amplitude=0.8,
    name="35_hz_tone",
)
high_tone = sine_wave(
    frequency_hz=180.0,
    duration_seconds=duration_seconds,
    sampling_rate_hz=sampling_rate_hz,
    amplitude=0.25,
    name="180_hz_tone",
)
impulses = impulse_train(
    duration_seconds=duration_seconds,
    sampling_rate_hz=sampling_rate_hz,
    impulse_rate_hz=8.0,
    amplitude=0.5,
    name="8_hz_impulses",
)

two_tone = add_signals([low_tone, high_tone], label="custom", name="two_tone_signal")
amplitude_modulated = multiply_signals(
    [
        low_tone,
        sine_wave(
            frequency_hz=3.0,
            duration_seconds=duration_seconds,
            sampling_rate_hz=sampling_rate_hz,
            amplitude=0.5,
            name="3_hz_modulator",
        ),
    ],
    label="custom",
    name="amplitude_modulated_signal",
)


impulse_response = transient_burst(
    duration_seconds=0.08,
    sampling_rate_hz=sampling_rate_hz,
    burst_frequency_hz=300.0,
    burst_start_seconds=0.0,
    burst_duration_seconds=0.04,
    amplitude=0.4,
    name="short_ringdown_kernel",
)
ringing_impulses = convolve_signals(
    impulses,
    impulse_response,
    mode="same",
    label="custom",
    name="ringing_impulse_train",
)


window = window_signal(
    duration_seconds=duration_seconds,
    sampling_rate_hz=sampling_rate_hz,
    window_duration_seconds=0.5,
    window_start_seconds=0.5,
    window_type="hann",
    name="hann",
)
windowed_two_tone = multiply_signals([window, two_tone], name="windowed_two_tone_signal")

custom_records = [two_tone, amplitude_modulated, ringing_impulses, windowed_two_tone]
for custom_record in custom_records:
    print(custom_record.name, custom_record.attributes)


In [ ]:
for custom_record in custom_records:
    plot_time_signal_adaptive(
        custom_record,
        start_seconds=0.0,
        duration_seconds=2.0,
        max_points=5000,
    )

## Static Time-Domain Plot

In [ ]:
record = transient_burst(duration_seconds=3.0, sampling_rate_hz=2000.0)
fig, ax = plot_time_signal(record, start_seconds=0.0, duration_seconds=1.0, max_points=None, downsample_method="envelope")

## Interactive Navigator

Use the slider and previous/next buttons to move through the signal. The standard matplotlib toolbar still provides zoom and pan.

In [ ]:
navigator = plot_time_signal_navigator(
    record,
    window_seconds=1.0,
)

## Try Different Synthetic Signals

In [ ]:
examples = [
    sine_wave(frequency_hz=30.0, duration_seconds=2.0),
    noisy_sine_wave(frequency_hz=30.0, noise_std=0.25, duration_seconds=2.0),
    impulse_train(duration_seconds=2.0, impulse_rate_hz=12.0),
    chirp_signal(duration_seconds=2.0),
    clipped_signal(duration_seconds=2.0),
]

for example in examples:
    plot_time_signal(example, duration_seconds=min(1.0, example.duration_seconds))